# Week 6 - Apache Spark: Architecture & Efficient Data Processing
**Dataset:** Sample Superstore

### Assignment Objectives
1. Understand Spark architecture (Driver, Cluster Manager, Executors) and execution modes  
2. Learn Lazy Evaluation and DAG (Lineage Graph)  
3. Read CSV / Parquet with proper schema handling  
4. Filter and select required columns  
5. Modify DataFrames (rename, cast, add columns)  
6. Apply transformations and actions correctly  
7. Understand wide transformations (Shuffle, Predicate Pushdown)  
8. Compare CSV vs Parquet performance  
9. Handle null values efficiently  
10. Build end-to-end data pipeline (read → transform → filter → write)  
11. Follow large-dataset best practices (show() over collect())
---

## Install and Set Up PySpark


In [2]:
 # !pip install pyspark --quiet

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("assignment - 6") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark is ready!")

Spark is ready!


---
## Step 1 - Spark Architecture


**Driver** - the main process that controls the application. It reads your code, builds a logical plan, and sends work to the cluster.

**Cluster Manager** - handles resource allocation. In local mode this is built into Spark itself. In production it would be YARN, Kubernetes, or Mesos.

**Executors** - JVM processes that do the actual computation. Each executor runs on a worker node and processes one or more partitions in parallel.

### Execution Modes

| Mode | Description |
|------|-------------|
| **Local** | Single JVM; ideal for dev/testing ( local[*] uses all cores ) |
| **Client** | Driver runs on the submitting machine; Executors on the cluster |
| **Cluster** | Driver runs inside the cluster; best for production |


---
## Step 2 - Read CSV with Explicit Schema

Spark can infer the schema automatically with inferSchema=True, but this triggers a full scan of the file before processing starts. For large files that wastes time.

Defining the schema upfront tells Spark exactly what to expect, skips the scan, and ensures the correct types from the start.

In [4]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)

csv_path = "/content/Sample - Superstore.csv"

schema = StructType([
    StructField("Row ID",        IntegerType(), True),
    StructField("Order ID",      StringType(),  True),
    StructField("Order Date",    StringType(),  True),
    StructField("Ship Date",     StringType(),  True),
    StructField("Ship Mode",     StringType(),  True),
    StructField("Customer ID",   StringType(),  True),
    StructField("Customer Name", StringType(),  True),
    StructField("Segment",       StringType(),  True),
    StructField("Country",       StringType(),  True),
    StructField("City",          StringType(),  True),
    StructField("State",         StringType(),  True),
    StructField("Postal Code",   StringType(),  True),
    StructField("Region",        StringType(),  True),
    StructField("Product ID",    StringType(),  True),
    StructField("Category",      StringType(),  True),
    StructField("Sub-Category",  StringType(),  True),
    StructField("Product Name",  StringType(),  True),
    StructField("Sales",         DoubleType(),  True),
    StructField("Quantity",      IntegerType(), True),
    StructField("Discount",      DoubleType(),  True),
    StructField("Profit",        DoubleType(),  True),
])

df = (
    spark.read
    .option("header", "true")
    .option("nullValue", "")
    .option("multiLine", "true")
    .option("escape", '"')
    .schema(schema)
    .csv(csv_path)
)

print(f"Rows      : {df.count()}")
print(f"Columns   : {len(df.columns)}")
print(f"Partitions: {df.rdd.getNumPartitions()}")
df.printSchema()
df.show(5, truncate=False)

Rows      : 9994
Columns   : 21
Partitions: 1
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-

---
## Step 3 - Lazy Evaluation and the DAG

Spark does not execute transformations immediately. When you call `.filter()` or `.select()`, Spark just records the instruction. Nothing actually runs until you call an **action** like `.show()`, `.count()`, or `.write()`.

This is called **lazy evaluation**. The benefit is that Spark can look at the entire chain of transformations at once and optimize them before running anything. This optimization process produces a **DAG** (Directed Acyclic Graph) a step-by-step execution plan.

### How Lazy Evaluation Works

```
Transformation 1  →  Transformation 2  →  Transformation N  →  ACTION
   (not executed)      (not executed)        (not executed)      (triggers execution)
```

- **Transformations** (e.g., `filter`, `select`, `withColumn`) are **lazy** — they build a DAG but do NOT run.  
- **Actions** (e.g., `show()`, `count()`, `write()`) **trigger** the DAG to execute.  
- Spark **optimizes the DAG** via the Catalyst Optimizer before running anything.


In [6]:

df_lazy = (
    df
    .filter(F.col("Sales") > 100)        # Transformation 1 - lazy
    .filter(F.col("Profit") > 0)         # Transformation 2 - lazy

    .select("Order ID", "Category", "Sales", "Profit")    # Transformation 3 - lazy

    .withColumn("Margin", F.round(F.col("Profit") / F.col("Sales") * 100, 2))   # Transformation 4 - lazy
)

# Calling explain() shows the DAG - still no execution
print("Logical plan (DAG):")
df_lazy.explain()

# This is the action — triggers execution of everything above at once
print("Results:")
df_lazy.show(5)


Logical plan (DAG):
== Physical Plan ==
*(1) Project [Order ID#1, Category#14, Sales#17, Profit#20, round(((Profit#20 / Sales#17) * 100.0), 2) AS Margin#174]
+- *(1) Filter (((isnotnull(Sales#17) AND isnotnull(Profit#20)) AND (Sales#17 > 100.0)) AND (Profit#20 > 0.0))
   +- FileScan csv [Order ID#1,Category#14,Sales#17,Profit#20] Batched: false, DataFilters: [isnotnull(Sales#17), isnotnull(Profit#20), (Sales#17 > 100.0), (Profit#20 > 0.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/Sample - Superstore.csv], PartitionFilters: [], PushedFilters: [IsNotNull(Sales), IsNotNull(Profit), GreaterThan(Sales,100.0), GreaterThan(Profit,0.0)], ReadSchema: struct<Order ID:string,Category:string,Sales:double,Profit:double>


Results:
+--------------+---------------+--------+-------+------+
|      Order ID|       Category|   Sales| Profit|Margin|
+--------------+---------------+--------+-------+------+
|CA-2016-152156|      Furniture|  261.96|41.9136|  16.0|
|CA-2016-152156|    

---
## Step 4 - Filtering and Selecting Columns

`filter()` removes rows that do not meet a condition. `select()` keeps only the columns you need. Both are **narrow transformations** each partition is processed independently with no data movement across the cluster.

In [7]:
df_tech = (
    df
    .filter(F.col("Category") == "Technology")
    .filter(F.col("Profit") > 0)
    .filter(F.col("Discount") < 0.3)
)

print(f"Technology rows with profit > 0 and discount < 30%: {df_tech.count()}")
df_tech.select("Order ID", "Product Name", "Sales", "Discount", "Profit").show(5, truncate=False)


Technology rows with profit > 0 and discount < 30%: 1543
+--------------+------------------------------------------------+--------+--------+--------+
|Order ID      |Product Name                                    |Sales   |Discount|Profit  |
+--------------+------------------------------------------------+--------+--------+--------+
|CA-2014-115812|Mitel 5320 IP Phone VoIP phone                  |907.152 |0.2     |90.7152 |
|CA-2014-115812|Konftel 250 Conference�phone�- Charcoal black   |911.424 |0.2     |68.3568 |
|CA-2014-143336|Cisco SPA 501G IP Phone                         |213.48  |0.2     |16.011  |
|CA-2016-121755|Imation�8GB Mini TravelDrive USB 2.0�Flash Drive|90.57   |0.0     |11.7741 |
|CA-2016-117590|GE 30524EE4                                     |1097.544|0.2     |123.4737|
+--------------+------------------------------------------------+--------+--------+--------+
only showing top 5 rows


In [8]:
# Select only the columns relevant for analysis
df_selected = df.select(
    "Order ID", "Order Date", "Ship Mode",
    "Customer Name", "Segment", "Region",
    "Category", "Sub-Category", "Sales", "Quantity", "Discount", "Profit"
)

print("Columns kept:", df_selected.columns)
df_selected.show(5, truncate=False)


Columns kept: ['Order ID', 'Order Date', 'Ship Mode', 'Customer Name', 'Segment', 'Region', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Discount', 'Profit']
+--------------+----------+--------------+---------------+---------+------+---------------+------------+--------+--------+--------+--------+
|Order ID      |Order Date|Ship Mode     |Customer Name  |Segment  |Region|Category       |Sub-Category|Sales   |Quantity|Discount|Profit  |
+--------------+----------+--------------+---------------+---------+------+---------------+------------+--------+--------+--------+--------+
|CA-2016-152156|11/8/2016 |Second Class  |Claire Gute    |Consumer |South |Furniture      |Bookcases   |261.96  |2       |0.0     |41.9136 |
|CA-2016-152156|11/8/2016 |Second Class  |Claire Gute    |Consumer |South |Furniture      |Chairs      |731.94  |3       |0.0     |219.582 |
|CA-2016-138688|6/12/2016 |Second Class  |Darrin Van Huff|Corporate|West  |Office Supplies|Labels      |14.62   |2       |0.0     |6

---
## Step 5 - Modifying the DataFrame

Three common operations:
- `withColumnRenamed()` removes spaces from column names, which avoids quoting issues later
- `cast()` changes the type of a column
- `withColumn()` adds a new column or replaces an existing one
.

In [9]:
df_clean = (
    df
    .withColumnRenamed("Row ID",        "row_id")
    .withColumnRenamed("Order ID",      "order_id")
    .withColumnRenamed("Order Date",    "order_date")
    .withColumnRenamed("Ship Date",     "ship_date")
    .withColumnRenamed("Ship Mode",     "ship_mode")
    .withColumnRenamed("Customer ID",   "customer_id")
    .withColumnRenamed("Customer Name", "customer_name")
    .withColumnRenamed("Segment",       "segment")
    .withColumnRenamed("City",          "city")
    .withColumnRenamed("State",         "state")
    .withColumnRenamed("Region",        "region")
    .withColumnRenamed("Product ID",    "product_id")
    .withColumnRenamed("Category",      "category")
    .withColumnRenamed("Sub-Category",  "sub_category")
    .withColumnRenamed("Product Name",  "product_name")
    .withColumnRenamed("Sales",         "sales")
    .withColumnRenamed("Quantity",      "quantity")
    .withColumnRenamed("Discount",      "discount")
    .withColumnRenamed("Profit",        "profit")
)

# Cast date strings to actual DateType
df_clean = (
    df_clean
    .withColumn("order_date", F.to_date(F.col("order_date"), "M/d/yyyy"))
    .withColumn("ship_date",  F.to_date(F.col("ship_date"),  "M/d/yyyy"))
)

df_clean = (
    df_clean
    .withColumn("profit_margin",  F.round(F.col("profit") / F.col("sales") * 100, 2))
    .withColumn("revenue_band",
        F.when(F.col("sales") >= 1000, "High")
         .when(F.col("sales") >= 300,  "Medium")
         .otherwise("Low"))
    .withColumn("is_profitable", F.col("profit") > 0)
    .withColumn("order_year",  F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("days_to_ship",
        F.datediff(F.col("ship_date"), F.col("order_date")))
)

print("Schema after modifications:")
df_clean.printSchema()
df_clean.select(
    "order_id", "category", "sales", "profit",
    "profit_margin", "revenue_band", "days_to_ship"
).show(5, truncate=False)


Schema after modifications:
root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- profit_margin: double (nullable = true)
 |-- revenue_band: string (nullable = false)
 |-- is_profitable: boolean (nul

---
## Step 6 - Transformations and Actions

Every Spark operation is either a transformation or an action.

**Transformations** are lazy — they build up the DAG but do not run. Examples: `filter`, `select`, `withColumn`, `groupBy`, `join`.

**Actions** trigger execution of the entire DAG. Examples: `show()`, `count()`, `write()`, `first()`.

The distinction matters because running multiple actions on the same DataFrame re-executes the full lineage each time.

In [10]:
# Transformations
df_transformed = (
    df_clean
    .filter(F.col("is_profitable") == True)
    .filter(F.col("discount") == 0)
    .select("order_id", "category", "region", "sales", "profit", "profit_margin")
)

# ACTION 1
print("Count of profitable orders with no discount:", df_transformed.count())

# Cache the result so the next action does not recompute
df_transformed.cache()

# ACTION 2 reads from cache instead of recomputing
df_transformed.show(5, truncate=False)



Count of profitable orders with no discount: 4768
+--------------+---------------+------+------+-------+-------------+
|order_id      |category       |region|sales |profit |profit_margin|
+--------------+---------------+------+------+-------+-------------+
|CA-2016-152156|Furniture      |South |261.96|41.9136|16.0         |
|CA-2016-152156|Furniture      |South |731.94|219.582|30.0         |
|CA-2016-138688|Office Supplies|West  |14.62 |6.8714 |47.0         |
|CA-2014-115812|Furniture      |West  |48.86 |14.1694|29.0         |
|CA-2014-115812|Office Supplies|West  |7.28  |1.9656 |27.0         |
+--------------+---------------+------+------+-------+-------------+
only showing top 5 rows


---
## Step 7 -Wide Transformations: Shuffle and Predicate Pushdown

**Wide transformations** (like groupBy, join, distinct) require data to move across partitions. This movement is called a **shuffle**. Shuffles are expensive because they involve disk I/O and network transfer.

Two ways to reduce their cost:
- **Filter before groupBy** — reduce the number of rows being shuffled
- **Predicate pushdown** — when reading Parquet files, Spark pushes filter conditions down to the file reader so it skips irrelevant row groups entirely

You can spot shuffles in the execution plan by looking for Exchange hashpartitioning.

In [11]:
category_summary = (
    df_clean
    .filter(F.col("sales") > 0)
    .groupBy("category", "region")
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("sales"),   2).alias("total_sales"),
        F.round(F.sum("profit"),  2).alias("total_profit"),
        F.round(F.avg("discount"),3).alias("avg_discount")
    )
    .orderBy("category", "region")
)

print("summary:")
category_summary.show(truncate=False)

print("Execution plan:")
category_summary.explain()


summary:
+---------------+-------+-----------+-----------+------------+------------+
|category       |region |order_count|total_sales|total_profit|avg_discount|
+---------------+-------+-----------+-----------+------------+------------+
|Furniture      |Central|481        |163797.16  |-2871.05    |0.297       |
|Furniture      |East   |601        |208291.2   |3046.17     |0.154       |
|Furniture      |South  |332        |117298.68  |6771.21     |0.122       |
|Furniture      |West   |707        |252612.74  |11504.95    |0.131       |
|Office Supplies|Central|1422       |167026.42  |8879.98     |0.253       |
|Office Supplies|East   |1712       |205516.05  |41014.58    |0.143       |
|Office Supplies|South  |995        |125651.31  |19986.39    |0.167       |
|Office Supplies|West   |1897       |220853.25  |52609.85    |0.093       |
|Technology     |Central|420        |170416.31  |33697.43    |0.133       |
|Technology     |East   |535        |264973.98  |47462.04    |0.143       |
|Te

## Window Functions in PySpark

A **Window Function** performs calculations across a group of related rows without combining them into a single row.

Unlike groupBy(), which returns one row per group, a window function keeps all the original rows and adds a new calculated column.

In this:

- partitionBy("category") divides the data into different categories.
- orderBy(F.col("sales").desc()) sorts the products within each category by sales in descending order.
- rank() assigns a rank based on the sales value in each category.


In [13]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("category").orderBy(F.col("sales").desc())

df_ranked = (
    df_clean
    .filter(F.col("profit") > 0)
    .withColumn("rank_in_category", F.rank().over(window_spec))
    .filter(F.col("rank_in_category") <= 3)
    .select(
        "category",
        "product_name",
        "sales",
        "profit",
        "rank_in_category"
    )
    .orderBy("category", "rank_in_category")
)

print("Top 3 sales per category (profitable orders only):")
df_ranked.show(truncate=False)

Top 3 sales per category (profitable orders only):
+---------------+-------------------------------------------------------------+---------+---------+----------------+
|category       |product_name                                                 |sales    |profit   |rank_in_category|
+---------------+-------------------------------------------------------------+---------+---------+----------------+
|Furniture      |Riverside Palais Royal Lawyers Bookcase, Royale Cherry Finish|4404.9   |1013.127 |1               |
|Furniture      |Riverside Palais Royal Lawyers Bookcase, Royale Cherry Finish|4228.704 |158.5764 |2               |
|Furniture      |HON 5400 Series Task Chairs for Big and Tall                 |3785.292 |420.588  |3               |
|Office Supplies|GBC Ibimaster 500 Manual ProClick Binding System             |9892.74  |4946.37  |1               |
|Office Supplies|Ibico EPK-21 Electric Binding System                         |9449.95  |4630.4755|2               |
|Office Suppl

---
## Step 8 - File Formats: CSV vs Parquet

CSV is a plain text row-based format. Every query must read every column, even ones you do not need.

Parquet is a binary columnar format. It stores data column by column, compresses it automatically, and embeds statistics (min/max per column block). This lets Spark skip entire chunks of data when filtering, which is called **predicate pushdown**.

For analytics workloads, Parquet is almost always the better choice.

In [14]:
import os, time

parquet_path = "/tmp/superstore.parquet"
output_csv   = "/tmp/superstore_output_csv"
output_parq  = "/tmp/superstore_output_parquet"

# Write to Parquet
df_clean.write.mode("overwrite").parquet(parquet_path)

# File size comparison
def folder_kb(path):
    total = sum(
        os.path.getsize(os.path.join(d, f))
        for d, _, files in os.walk(path) for f in files
    )
    return round(total / 1024, 1)

csv_kb  = round(os.path.getsize("/content/Sample - Superstore.csv") / 1024, 1)
parq_kb = folder_kb(parquet_path)

print(f"CSV size     : {csv_kb} KB")
print(f"Parquet size : {parq_kb} KB")
print(f"Reduction    : {round((1 - parq_kb/csv_kb)*100, 1)}%")


CSV size     : 2234.2 KB
Parquet size : 459.0 KB
Reduction    : 79.5%


In [16]:
t0 = time.time()
spark.read.option("header","true").schema(schema).csv("/content/Sample - Superstore.csv").count()
csv_time = round(time.time() - t0, 3)

t0 = time.time()
spark.read.parquet(parquet_path).count()
parq_time = round(time.time() - t0, 3)

print(f"CSV read time: {csv_time}s")
print(f"Parquet read time : {parq_time}s")

df_parquet_filtered = (
    spark.read.parquet(parquet_path)
    .filter(F.col("category") == "Technology")
    .select("order_id", "sales", "profit")
)
print("\nParquet execution plan:")
df_parquet_filtered.explain()


CSV read time: 0.263s
Parquet read time : 0.345s

Parquet execution plan:
== Physical Plan ==
*(1) Project [order_id#921, sales#937, profit#940]
+- *(1) Filter (isnotnull(category#934) AND (category#934 = Technology))
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#921,category#934,sales#937,profit#940] Batched: true, DataFilters: [isnotnull(category#934), (category#934 = Technology)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/superstore.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(category), EqualTo(category,Technology)], ReadSchema: struct<order_id:string,category:string,sales:double,profit:double>




---
## Step 9 - Handling Null Values

Nulls in real datasets can cause silent errors — aggregations skip them, joins can produce unexpected results, and type casts may silently return null.

Common strategies:
- isNull() / isNotNull() - identify nulls
- na.drop() - remove rows where specified columns are null
- na.fill() - replace nulls with a constant


In [17]:
# Check nulls across all columns
print("Null count per column:")
df_clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
]).show()


Null count per column:
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------------+------------+-------------+----------+-----------+------------+
|row_id|order_id|order_date|ship_date|ship_mode|customer_id|customer_name|segment|Country|city|state|Postal Code|region|product_id|category|sub_category|product_name|sales|quantity|discount|profit|profit_margin|revenue_band|is_profitable|order_year|order_month|days_to_ship|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+-------------+------------+-------------+----------+-----------+------------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0

In [18]:
df_null_demo = df_clean.withColumn(
    "profit_margin",
    F.when(F.col("sales") == 0, None)
     .otherwise(F.col("profit_margin"))
)

# Strategy 1 - fill nulls with 0
df_filled = df_null_demo.na.fill({"profit_margin": 0.0})

# Strategy 2 - filter out rows where critical fields are null
df_dropped = df_null_demo.na.drop(subset=["sales", "profit"])

# Strategy 3 - conditional replacement
df_conditional = df_null_demo.withColumn(
    "profit_margin",
    F.when(F.col("profit_margin").isNull(), 0.0)
     .otherwise(F.col("profit_margin"))
)

print(f"Rows before drop : {df_null_demo.count()}")
print(f"Rows after drop  : {df_dropped.count()}")
print("Fill approach sample:")
df_filled.select("order_id", "sales", "profit", "profit_margin").show(5)


Rows before drop : 9994
Rows after drop  : 9994
Fill approach sample:
+--------------+--------+--------+-------------+
|      order_id|   sales|  profit|profit_margin|
+--------------+--------+--------+-------------+
|CA-2016-152156|  261.96| 41.9136|         16.0|
|CA-2016-152156|  731.94| 219.582|         30.0|
|CA-2016-138688|   14.62|  6.8714|         47.0|
|US-2015-108966|957.5775|-383.031|        -40.0|
|US-2015-108966|  22.368|  2.5164|        11.25|
+--------------+--------+--------+-------------+
only showing top 5 rows


---
## Step 10 - End-to-End Data Pipeline

A data pipeline connects all the previous steps in sequence:
### Pipeline Flow
```
CSV (raw)
  → Read with schema
  → Handle nulls
  → Cast / rename columns
  → Add derived columns
  → Filter business rules
  → Aggregate
  → Write Parquet + CSV

This is the structure used in real ETL jobs.

In [27]:
output_csv  = "/tmp/superstore_output_csv"
output_parq = "/tmp/superstore_output_parquet"

print("PIPELINE START")

# 1. Read
print(" \n1. Reading raw CSV...")
df_pipe = (
    spark.read
    .option("header", "true")
    .option("nullValue", "")
    .option("multiLine", "true")
    .option("escape", '"')
    .schema(schema)
    .csv("/content/Sample - Superstore.csv")
)
print(f"   {df_pipe.count()} rows loaded")

# 2. Rename columns
print("2. Renaming columns")
rename_map = {
    "Row ID": "row_id", "Order ID": "order_id", "Order Date": "order_date",
    "Ship Date": "ship_date", "Ship Mode": "ship_mode", "Customer ID": "customer_id",
    "Customer Name": "customer_name", "Segment": "segment", "City": "city",
    "State": "state", "Region": "region", "Product ID": "product_id",
    "Category": "category", "Sub-Category": "sub_category",
    "Product Name": "product_name", "Sales": "sales",
    "Quantity": "quantity", "Discount": "discount", "Profit": "profit"
}
for old, new in rename_map.items():
    df_pipe = df_pipe.withColumnRenamed(old, new)

# 3. Cast and enrich
print("3. Casting types and adding derived columns")
df_pipe = (
    df_pipe
    .withColumn("order_date",    F.to_date("order_date", "M/d/yyyy"))
    .withColumn("ship_date",     F.to_date("ship_date",  "M/d/yyyy"))
    .withColumn("profit_margin", F.round(F.col("profit") / F.col("sales") * 100, 2))
    .withColumn("days_to_ship",  F.datediff(F.col("ship_date"), F.col("order_date")))
    .withColumn("revenue_band",
        F.when(F.col("sales") >= 1000, "High")
         .when(F.col("sales") >= 300,  "Medium")
         .otherwise("Low"))
    .withColumn("order_year", F.year("order_date"))
)

# 4. Handle nulls
print("4. Handling nulls")
df_pipe = df_pipe.na.fill({"discount": 0.0, "profit": 0.0})

# 5. Filter
print("5. Applying business filter")
df_pipe = (
    df_pipe
    .filter(F.col("profit") > 0)
    .filter(F.col("discount") <= 0.3)
)
print(f"   {df_pipe.count()} rows after filter")

# 6. Write
print("6. Writing outputs")
df_pipe.write.mode("overwrite").partitionBy("category").parquet(output_parq)
df_pipe.coalesce(1).write.mode("overwrite").option("header","true").csv(output_csv)

print("\n PIPELINE COMPLETE")
print(f"Parquet : {output_parq}")
print(f"CSV     : {output_csv}")

PIPELINE START
 
1. Reading raw CSV...
   9994 rows loaded
2. Renaming columns
3. Casting types and adding derived columns
4. Handling nulls
5. Applying business filter
   8032 rows after filter
6. Writing outputs

 PIPELINE COMPLETE
Parquet : /tmp/superstore_output_parquet
CSV     : /tmp/superstore_output_csv


---
## Step 11 - Best Practices for Large Datasets



In [20]:
df_clean.show(5)

# count() triggers a full scan
print("Total rows:", df_clean.count())

# cache() stores a DataFrame in memory after the first action
df_clean.cache()
print("First count  (builds cache):", df_clean.count())
print("Second count (from cache)  :", df_clean.count())
df_clean.unpersist()

# coalesce() reduces small output files without a full shuffle
df_clean.coalesce(1).write.mode("overwrite").option("header","true").csv("/tmp/single_output")
print("Single-file CSV written")

# select() early
df_lean = df_clean.select("order_id", "category", "region", "sales", "profit")

# explain() helps you spot problems before running the full job
print("\nExecution plan:")
df_lean.groupBy("category").agg(F.sum("sales")).explain()

print("\nBest practices summary:")
print("  show()        instead of collect() for previewing")
print("  cache()       when reusing a DataFrame across multiple actions")
print("  filter()      before groupBy or join to reduce shuffle volume")
print("  select()      early to drop columns you do not need")
print("  Parquet       for all intermediate and final storage")
print("  partitionBy() on write for tables queried by a known column")
print("  coalesce()    to avoid writing hundreds of tiny files")
print("  explain()     to catch shuffles and full scans before they hit production")


+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------+------------+-------------+----------+-----------+------------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|  customer_name|  segment|      Country|           city|     state|Postal Code|region|     product_id|       category|sub_category|        product_name|   sales|quantity|discount|  profit|profit_margin|revenue_band|is_profitable|order_year|order_month|days_to_ship|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------+------------+-------------+----------+-----------+

---
## Summary

| Topic | Key Point |
|-------|-----------|
| Architecture | Driver plans, Executors compute, Cluster Manager allocates resources |
| Lazy evaluation | Transformations build the DAG; actions trigger execution |
| Schema | Define it explicitly to avoid an extra file scan on read |
| Narrow transformations | filter, select — no shuffle, process each partition independently |
| Wide transformations | groupBy, join — shuffle required, data moves across partitions |
| Predicate pushdown | Parquet pushes filters to the file reader, skipping unneeded row groups |
| Null handling | Decide per column: drop, fill with constant, or fill conditionally |
| CSV vs Parquet | Parquet is smaller, faster to read, and supports predicate pushdown |
| collect() | Pulls all data to the driver — avoid on large datasets |
| cache() | Prevents recomputation when the same DataFrame is used multiple times |
|


In [32]:
import os
from google.colab import files

# Find the actual filename
filename = [f for f in os.listdir(output_csv) if f.endswith(".csv")][0]
print("File:", filename)

files.download(f"{output_csv}/{filename}")

File: part-00000-c4775f88-3c73-4060-9e94-e4326309c03e-c000.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>